In [21]:
%%capture
%pip install langgraph langchain-core langchain-openai presidio-analyzer presidio-anonymizer ipywidgets 2>/dev/null
%python -m spacy download en_core_web_sm -q 2>/dev/null
print("Ready")

In [22]:
from dotenv import load_dotenv

load_dotenv()

True

In [23]:
import re, hashlib, logging
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("GUARD")

_analyzer   = AnalyzerEngine()
_anonymizer = AnonymizerEngine()

# Layer 0: Nepal IBAN
_IBAN_RE = re.compile(r"\b(NP\d{2}[A-Z]{2,6}\d{6,18})\b", re.IGNORECASE)

# Layer 1: Context keyword + digit block
_CTX_RE = re.compile(
    r"""
    (?:
        account \s* (?:number|no|num|nmbr|n[o0]\.?|[\x23]) |
        acc(?:ount)? \s* [\x23]                              |
        acc(?:ount)? \s* (?:no|num|number)                   |
        a/c  \s* (?:no|num|number)?                          |
        khata \s* (?:number|no|num|nambor|nambur)?           |
        ac\s*no\.?                                           |
        (?:for\s+)?account\s+(?=[A-Z]{0,6}\d)
    )
    \s* [:\-\x23]? \s*
    ([A-Z]{0,6}\d[\d\s\-]{7,24}\d)
    """,
    re.VERBOSE | re.IGNORECASE
)

# Layer 2: Alphanumeric prefix (e.g. LSB0012345678)
_ALPHA_RE = re.compile(r"\b([A-Z]{2,6}\d{6,16})\b")

# Layer 3: Standalone digit run (9-15 digits)
_DIGIT_RE = re.compile(r"(?<!\d)([\d][\d\s\-]{7,16}[\d])(?!\d)")


def _nd(s):
    return re.sub(r"\D", "", s)


def _valid(raw, norm):
    if not norm.isdigit(): return False
    l = len(norm)
    if not (9 <= l <= 15): return False
    if l == 10 and norm[:2] in ("97","98","96","01","02","06"): return False
    if l == 9  and norm[:2] in ("01","02","06"): return False
    return True

def find_spans(text):
    spans = []
    for m in _IBAN_RE.finditer(text):
        spans.append(m.span(1))
    for m in _CTX_RE.finditer(text):
        s, e = m.span(1)
        if not any(cs<=s and e<=ce for cs,ce in spans):
            spans.append((s, e))
    for m in _ALPHA_RE.finditer(text):
        s, e = m.span()
        if not any(cs<=s and e<=ce for cs,ce in spans):
            spans.append((s, e))
    for m in _DIGIT_RE.finditer(text):
        raw = m.group(1)
        norm = _nd(raw)
        if _valid(raw, norm):
            s, e = m.span(1)
            if not any(cs<=s and e<=ce for cs,ce in spans):
                spans.append((s, e))
    if not spans:
        return []
    spans.sort()
    merged = [spans[0]]
    for s, e in spans[1:]:
        ps, pe = merged[-1]
        if s <= pe:
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s, e))
    return merged

#Token Creation
def _make_token(original, token_map):
    tok = "[ACCOUNT_NUMBER_" + hashlib.md5(original.encode()).hexdigest()[:6].upper() + "]" # hasihing generates unique id
    token_map[tok] = original # Store token mapping
    return tok

#Find spans, replace account numbers with tokens and log each replacement
def tokenize_accounts(text, token_map):
    for s, e in reversed(find_spans(text)):
        orig = text[s:e].strip()
        tok  = _make_token(orig, token_map)
        text = text[:s] + tok + text[e:]
        log.info(f"[TOKENIZED]  {orig!r}  ->  {tok}")
    return text

#Token Pattern
_TPAT = re.compile(r"\[[A-Z_]+_[0-9A-F]{6}(?:_\d+)?\]")

#Prevents leakage, keeps query meaningful
def restore_for_rag(sanitized, token_map):
    return _TPAT.sub(
        lambda m: "account balance"
                  if m.group(0).startswith("[ACCOUNT_NUMBER_")
                  else m.group(0),
        sanitized
    )


def extract_account_number(token_map):
    """THE ONLY sanctioned access point. Called only by cbs_node."""
    for tok, orig in token_map.items():
        if tok.startswith("[ACCOUNT_NUMBER_"):
            return orig.strip()
    return None


def run_guardrail(user_input):
    #Initialize token map
    token_map = {}
    log.info(f"[INPUT]     {user_input!r}")

    #Tokenize account numbers
    sanitized = tokenize_accounts(user_input, token_map)

    #Presidio PII Detection
    hits = _analyzer.analyze(
        text=sanitized,
        language="en",
        entities=["PHONE_NUMBER", "EMAIL_ADDRESS", "CREDIT_CARD"],
    )
    if hits:
        # Anonymize PII
        anon = _anonymizer.anonymize(
            text=sanitized,
            analyzer_results=hits,
            operators={
                "PHONE_NUMBER":  OperatorConfig("replace", {"new_value": "[PHONE]"}),
                "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "[EMAIL]"}),
                "CREDIT_CARD":   OperatorConfig("replace", {"new_value": "[CARD]"}),
            },
        )
        sanitized = anon.text

    result = {
        "llm_query": sanitized,
        "rag_query": restore_for_rag(sanitized, token_map),
        "token_map": token_map,
    }
    log.info(f"[LLM SEES]  {result['llm_query']!r}")
    log.info(f"[RAG SEES]  {result['rag_query']!r}")
    if token_map:
        log.info(f"[TOKEN MAP] {token_map}  <- real number, memory only")
    else:
        log.info("[TOKEN MAP] {}  (no account number detected)")
    return result


# Smoke test
print("\nGuard loaded\n")
print("=" * 55)
g = run_guardrail("My account number is 1234567890. What is my balance?")
print()
print(f"  token_map : {g['token_map']}")
print(f"  LLM sees  : {g['llm_query']!r}")
print(f"  RAG sees  : {g['rag_query']!r}")
print()
print("  => 1234567890 is ONLY in token_map, nowhere else")

17:48:48  INFO     nlp_engine not provided, creating default.


17:48:57  INFO     Created NLP engine: spacy. Loaded models: ['en']
17:48:57  INFO     registry not provided, creating default.
17:48:57  INFO     Loaded recognizer: CreditCardRecognizer
17:48:57  INFO     Loaded recognizer: CreditCardRecognizer
17:48:57  INFO     Loaded recognizer: CreditCardRecognizer
17:48:57  INFO     Loaded recognizer: CreditCardRecognizer
17:48:57  INFO     Loaded recognizer: UsBankRecognizer
17:48:57  INFO     Loaded recognizer: UsLicenseRecognizer
17:48:57  INFO     Loaded recognizer: UsItinRecognizer
17:48:57  INFO     Loaded recognizer: UsPassportRecognizer
17:48:57  INFO     Loaded recognizer: UsSsnRecognizer
17:48:57  INFO     Loaded recognizer: NhsRecognizer
17:48:57  INFO     Loaded recognizer: EsNifRecognizer
17:48:57  INFO     Loaded recognizer: EsNieRecognizer
17:48:57  INFO     Loaded recognizer: ItDriverLicenseRecognizer
17:48:57  INFO     Loaded recognizer: ItFiscalCodeRecognizer
17:48:57  INFO     Loaded recognizer: ItVatCodeRecognizer
17:48:57  IN


Guard loaded


  token_map : {'[ACCOUNT_NUMBER_E807F1]': '1234567890'}
  LLM sees  : 'My account number is [ACCOUNT_NUMBER_E807F1]. What is my balance?'
  RAG sees  : 'My account number is account balance. What is my balance?'

  => 1234567890 is ONLY in token_map, nowhere else


In [24]:
from mock_cbs_api import app
import threading
import uvicorn
import time

# Start the FastAPI app in a background thread
config = uvicorn.Config(app, host="0.0.0.0", port=8001, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

# Wait for the server to start
time.sleep(2)
print("CBS API running on port 8001")

INFO:     Started server process [25987]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8001): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


CBS API running on port 8001


In [26]:
import requests

CBS_BASE = "http://localhost:8001/api/v1"


def cbs_lookup(account_number):
    norm = re.sub(r"[\s\-]", "", account_number).upper()
    try:
        resp = requests.get(f"{CBS_BASE}/account/{norm}", timeout=5)
        if resp.status_code == 200:
            return {"found": True, "data": resp.json()}
        elif resp.status_code == 404:
            return {"found": False, "data": None}
        else:
            resp.raise_for_status()
    except requests.RequestException as exc:
        log.error(f"[CBS ERROR] {exc}")
        return {"found": False, "data": None}


def format_balance(account_number):
    log.info(f"[CBS CALL]  cbs_lookup('{account_number[:4]}****')")
    res = cbs_lookup(account_number)
    if not res["found"]:
        return "Account not found. Please check the number or visit your nearest branch."
    d   = res["data"]
    cur = "Rs." if d["currency"] == "NPR" else d["currency"]
    ico = {"Active": "Active", "Dormant": "DORMANT"}.get(d["status"], d["status"])
    out = [
        "### Account Details",
        f"**Holder:** {d['account_holder']}",
        f"**Type:** {d['account_type']} — {d['product_name']}",
        f"**Branch:** {d['branch']}",
        f"**Status:** {ico}",
        "",
        "### Balance",
        f"**Current:** {cur} {d['balance']:,.2f}",
        f"**Available:** {cur} {d['available_balance']:,.2f}",
    ]
    extra = d.get("extra") or {}
    if "maturity_date" in extra:
        out += ["", "### Fixed Deposit Info",
                f"**Maturity:** {extra['maturity_date']}",
                f"**Rate:** {extra['interest_rate']}% p.a."]
    if "loan_outstanding" in extra:
        out += ["", "### Loan Info",
                f"**Outstanding:** {cur} {extra['loan_outstanding']:,.2f}",
                f"**Next EMI:** {extra['next_emi_date']}",
                f"**EMI:** {cur} {extra['emi_amount']:,.2f}"]
    if d["status"] == "Dormant":
        out.append("Account is DORMANT. Visit branch to reactivate.")
    log.info(f"[CBS OK]    {d['account_holder']} | {d['account_type']} | {cur} {d['balance']:,.2f}")
    return "\n".join(out)


# Quick connectivity check
_ping = requests.get("http://localhost:8001/health", timeout=3).json()
print("CBS API health:", _ping)
print("CBS lookup ready — HTTP-backed via mock_cbs_api")

CBS API health: {'status': 'ok', 'service': 'Mock CBS API'}
CBS lookup ready — HTTP-backed via mock_cbs_api


In [27]:
from langchain_openai import ChatOpenAI
import os

# Ensure llm is initialized correctly
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=os.getenv("OPENAI_API_KEY"))
print("LLM ready:", type(llm))

LLM ready: <class 'langchain_openai.chat_models.base.ChatOpenAI'>


In [28]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Debugging to ensure llm is not overwritten
print(f"Type of llm before use: {type(llm)}")
assert isinstance(llm, ChatOpenAI), "llm is an instance of ChatOpenAI"

class BankState(TypedDict):
    user_input : str
    llm_query  : str
    rag_query  : str
    token_map  : dict
    intent     : str
    response   : str
    audit_log  : list

def guard_node(state: BankState) -> BankState:
    log.info("=" * 50)
    log.info("[NODE] guard_node  START")
    r     = run_guardrail(state["user_input"])
    audit = state.get("audit_log", [])
    audit.append({
        "node"      : "guard_node",
        "user_input": state["user_input"],
        "llm_query" : r["llm_query"],
        "rag_query" : r["rag_query"],
        "token_map" : dict(r["token_map"]),
        "note"      : "DETERMINISTIC — zero LLM. Account number in token_map only.",
    })
    log.info("[NODE] guard_node  DONE")
    return {
        **state,
        "llm_query": r["llm_query"],
        "rag_query": r["rag_query"],
        "token_map": r["token_map"],
        "audit_log": audit,
    }


def route(state: BankState) -> Literal["cbs_node", "llm_node"]:
    decision = "cbs_node" if state["token_map"] else "llm_node"
    log.info(f"[ROUTER] token_map={bool(state['token_map'])}  =>  {decision}")
    return decision


def cbs_node(state: BankState) -> BankState:
    log.info("[NODE] cbs_node  START")
    acct  = extract_account_number(state["token_map"])
    audit = state.get("audit_log", [])
    if not acct:
        resp = "Please provide your account number. Example: My account number is 1234567890"
        audit.append({"node": "cbs_node", "note": "No account number in token_map"})
    else:
        resp = format_balance(acct)
        audit.append({
            "node"    : "cbs_node",
            "note"    : "Retrieved real number from token_map -> called CBS. NOT from LLM query.",
            "cbs_arg" : acct[:4] + "*" * (len(acct) - 4),
        })
    log.info("[NODE] cbs_node  DONE")
    return {**state, "intent": "account_inquiry", "response": resp, "audit_log": audit}


def llm_node(state: BankState) -> BankState:
    log.info("[NODE] llm_node  START")
    log.info(f"[LLM INPUT] {state['llm_query']!r}")
    msgs = [
        SystemMessage(content=(
            "You are a helpful Laxmi Sunrise Bank assistant. "
            "Answer general banking questions in English, under 80 words."
        )),
        HumanMessage(content=state["llm_query"]),
    ]
    r     = llm.invoke(msgs)
    audit = state.get("audit_log", [])
    audit.append({
        "node"    : "llm_node",
        "llm_saw" : state["llm_query"],
        "note"    : "LLM received llm_query (sanitized). No account number. No real PII.",
    })
    log.info(f"[LLM OUTPUT] {r.content[:80]!r}")
    log.info("[NODE] llm_node  DONE")
    return {**state, "intent": "general", "response": r.content, "audit_log": audit}


b = StateGraph(BankState)
b.add_node("guard_node", guard_node)
b.add_node("cbs_node",   cbs_node)
b.add_node("llm_node",   llm_node)
b.set_entry_point("guard_node")
b.add_conditional_edges(
    "guard_node",
    route,
    {"cbs_node": "cbs_node", "llm_node": "llm_node"},
)
b.add_edge("cbs_node", END)
b.add_edge("llm_node", END)
graph = b.compile()

print("\nLangGraph compiled — 2 nodes, deterministic routing, no tool binding\n")
graph.get_graph().print_ascii()

Type of llm before use: <class 'langchain_openai.chat_models.base.ChatOpenAI'>

LangGraph compiled — 2 nodes, deterministic routing, no tool binding

           +-----------+             
           | __start__ |             
           +-----------+             
                  *                  
                  *                  
                  *                  
          +------------+             
          | guard_node |             
          +------------+             
           ...        ...            
          .              .           
        ..                ..         
+----------+           +----------+  
| cbs_node |           | llm_node |  
+----------+           +----------+  
           ***        ***            
              *      *               
               **  **                
            +---------+              
            | __end__ |              
            +---------+              


In [29]:
def run(query, show_audit=True):
    print()
    print("=" * 60)
    print(f"USER: {query}")
    print("=" * 60)
    r = graph.invoke({
        "user_input" : query,
        "llm_query"  : "",
        "rag_query"  : "",
        "token_map"  : {},
        "intent"     : "",
        "response"   : "",
        "audit_log"  : [],
    })
    if show_audit:
        print("\n  AUDIT LOG — what each node saw:")
        print("  " + "-" * 56)
        for step in r["audit_log"]:
            print(f"  [{step['node']}]")
            for k, v in step.items():
                if k == "node":
                    continue
                if k == "token_map":
                    if v:
                        tok, real = list(v.items())[0]
                        print(f'    token_map : {tok}  ->  "{real}"  <- REAL NUMBER')
                    else:
                        print("    token_map : {}  (no account number detected)")
                elif k == "note":
                    print(f"    // {v}")
                else:
                    print(f"    {k} : {str(v)[:90]}")
    print("\n  RESPONSE:")
    for line in r["response"].split("\n"):
        print(f"  {line}")
    print()
    return r

In [12]:
_ = run("My account number is 1234567890. What is my balance?")

14:08:50  INFO     ==================================================
14:08:50  INFO     [NODE] guard_node  START
14:08:50  INFO     [INPUT]     'My account number is 1234567890. What is my balance?'
14:08:50  INFO     [TOKENIZED]  '1234567890'  ->  [ACCOUNT_NUMBER_E807F1]



USER: My account number is 1234567890. What is my balance?


14:08:50  INFO     [LLM SEES]  'My account number is [ACCOUNT_NUMBER_E807F1]. What is my balance?'
14:08:50  INFO     [RAG SEES]  'My account number is account balance. What is my balance?'
14:08:50  INFO     [TOKEN MAP] {'[ACCOUNT_NUMBER_E807F1]': '1234567890'}  <- real number, memory only
14:08:50  INFO     [NODE] guard_node  DONE
14:08:50  INFO     [ROUTER] token_map=True  =>  cbs_node
14:08:50  INFO     [NODE] cbs_node  START
14:08:50  INFO     [CBS CALL]  cbs_lookup('1234****')
14:08:50  INFO     [CBS OK]    Ram Prasad Sharma | Savings | Rs. 125,430.50
14:08:50  INFO     [NODE] cbs_node  DONE



  AUDIT LOG — what each node saw:
  --------------------------------------------------------
  [guard_node]
    user_input : My account number is 1234567890. What is my balance?
    llm_query : My account number is [ACCOUNT_NUMBER_E807F1]. What is my balance?
    rag_query : My account number is account balance. What is my balance?
    token_map : [ACCOUNT_NUMBER_E807F1]  ->  "1234567890"  <- REAL NUMBER
    // DETERMINISTIC — zero LLM. Account number in token_map only.
  [cbs_node]
    // Retrieved real number from token_map -> called CBS. NOT from LLM query.
    cbs_arg : 1234******

  RESPONSE:
  ### Account Details
  **Holder:** Ram Prasad Sharma
  **Type:** Savings — Sunaulo Bachat Khata
  **Branch:** Kathmandu Main Branch
  **Status:** Active
  
  ### Balance
  **Current:** Rs. 125,430.50
  **Available:** Rs. 120,430.50



In [13]:
_ = run("mero khata number 007123456789 ko balance kati cha?")

14:08:50  INFO     ==================================================
14:08:50  INFO     [NODE] guard_node  START
14:08:50  INFO     [INPUT]     'mero khata number 007123456789 ko balance kati cha?'
14:08:50  INFO     [TOKENIZED]  '007123456789'  ->  [ACCOUNT_NUMBER_E81E11]
14:08:50  INFO     [LLM SEES]  'mero khata number [ACCOUNT_NUMBER_E81E11] ko balance kati cha?'
14:08:50  INFO     [RAG SEES]  'mero khata number account balance ko balance kati cha?'
14:08:50  INFO     [TOKEN MAP] {'[ACCOUNT_NUMBER_E81E11]': '007123456789'}  <- real number, memory only
14:08:50  INFO     [NODE] guard_node  DONE
14:08:50  INFO     [ROUTER] token_map=True  =>  cbs_node
14:08:50  INFO     [NODE] cbs_node  START
14:08:50  INFO     [CBS CALL]  cbs_lookup('0071****')



USER: mero khata number 007123456789 ko balance kati cha?


14:08:50  INFO     [CBS OK]    Anita Gurung | Savings | Rs. 18,750.00
14:08:50  INFO     [NODE] cbs_node  DONE



  AUDIT LOG — what each node saw:
  --------------------------------------------------------
  [guard_node]
    user_input : mero khata number 007123456789 ko balance kati cha?
    llm_query : mero khata number [ACCOUNT_NUMBER_E81E11] ko balance kati cha?
    rag_query : mero khata number account balance ko balance kati cha?
    token_map : [ACCOUNT_NUMBER_E81E11]  ->  "007123456789"  <- REAL NUMBER
    // DETERMINISTIC — zero LLM. Account number in token_map only.
  [cbs_node]
    // Retrieved real number from token_map -> called CBS. NOT from LLM query.
    cbs_arg : 0071********

  RESPONSE:
  ### Account Details
  **Holder:** Anita Gurung
  **Type:** Savings — Shakti Bachat Khata
  **Branch:** Lalitpur Branch
  **Status:** DORMANT
  
  ### Balance
  **Current:** Rs. 18,750.00
  **Available:** Rs. 18,750.00
  Account is DORMANT. Visit branch to reactivate.



In [14]:
_ = run("Please check acc no LSB0012345678")


USER: Please check acc no LSB0012345678


14:08:50  INFO     ==================================================
14:08:50  INFO     [NODE] guard_node  START
14:08:50  INFO     [INPUT]     'Please check acc no LSB0012345678'
14:08:50  INFO     [TOKENIZED]  'LSB0012345678'  ->  [ACCOUNT_NUMBER_789499]
14:08:51  INFO     [LLM SEES]  'Please check acc no [ACCOUNT_NUMBER_789499]'
14:08:51  INFO     [RAG SEES]  'Please check acc no account balance'
14:08:51  INFO     [TOKEN MAP] {'[ACCOUNT_NUMBER_789499]': 'LSB0012345678'}  <- real number, memory only
14:08:51  INFO     [NODE] guard_node  DONE
14:08:51  INFO     [ROUTER] token_map=True  =>  cbs_node
14:08:51  INFO     [NODE] cbs_node  START
14:08:51  INFO     [CBS CALL]  cbs_lookup('LSB0****')
14:08:51  INFO     [CBS OK]    Prakash Bahadur Magar | Savings | USD 3,210.75
14:08:51  INFO     [NODE] cbs_node  DONE



  AUDIT LOG — what each node saw:
  --------------------------------------------------------
  [guard_node]
    user_input : Please check acc no LSB0012345678
    llm_query : Please check acc no [ACCOUNT_NUMBER_789499]
    rag_query : Please check acc no account balance
    token_map : [ACCOUNT_NUMBER_789499]  ->  "LSB0012345678"  <- REAL NUMBER
    // DETERMINISTIC — zero LLM. Account number in token_map only.
  [cbs_node]
    // Retrieved real number from token_map -> called CBS. NOT from LLM query.
    cbs_arg : LSB0*********

  RESPONSE:
  ### Account Details
  **Holder:** Prakash Bahadur Magar
  **Type:** Savings — NRN Savings
  **Branch:** New Road Branch
  **Status:** Active
  
  ### Balance
  **Current:** USD 3,210.75
  **Available:** USD 3,210.75



In [15]:
_ = run("What is the status of acc no 1122334455667?")

14:08:51  INFO     ==================================================
14:08:51  INFO     [NODE] guard_node  START
14:08:51  INFO     [INPUT]     'What is the status of acc no 1122334455667?'
14:08:51  INFO     [TOKENIZED]  '1122334455667'  ->  [ACCOUNT_NUMBER_F09BA3]
14:08:51  INFO     [LLM SEES]  'What is the status of acc no [ACCOUNT_NUMBER_F09BA3]?'
14:08:51  INFO     [RAG SEES]  'What is the status of acc no account balance?'
14:08:51  INFO     [TOKEN MAP] {'[ACCOUNT_NUMBER_F09BA3]': '1122334455667'}  <- real number, memory only
14:08:51  INFO     [NODE] guard_node  DONE
14:08:51  INFO     [ROUTER] token_map=True  =>  cbs_node
14:08:51  INFO     [NODE] cbs_node  START
14:08:51  INFO     [CBS CALL]  cbs_lookup('1122****')


14:08:51  INFO     [CBS OK]    Bikash Kumar Rai | Fixed Deposit | Rs. 500,000.00
14:08:51  INFO     [NODE] cbs_node  DONE



USER: What is the status of acc no 1122334455667?

  AUDIT LOG — what each node saw:
  --------------------------------------------------------
  [guard_node]
    user_input : What is the status of acc no 1122334455667?
    llm_query : What is the status of acc no [ACCOUNT_NUMBER_F09BA3]?
    rag_query : What is the status of acc no account balance?
    token_map : [ACCOUNT_NUMBER_F09BA3]  ->  "1122334455667"  <- REAL NUMBER
    // DETERMINISTIC — zero LLM. Account number in token_map only.
  [cbs_node]
    // Retrieved real number from token_map -> called CBS. NOT from LLM query.
    cbs_arg : 1122*********

  RESPONSE:
  ### Account Details
  **Holder:** Bikash Kumar Rai
  **Type:** Fixed Deposit — Samman Bachat Khata
  **Branch:** Biratnagar Branch
  **Status:** Active
  
  ### Balance
  **Current:** Rs. 500,000.00
  **Available:** Rs. 0.00
  
  ### Fixed Deposit Info
  **Maturity:** 2025-01-10
  **Rate:** 10.5% p.a.



In [16]:
_ = run("account: 0101234567890 — home loan outstanding?")

14:08:51  INFO     ==================================================
14:08:51  INFO     [NODE] guard_node  START
14:08:51  INFO     [INPUT]     'account: 0101234567890 — home loan outstanding?'
14:08:51  INFO     [TOKENIZED]  '0101234567890'  ->  [ACCOUNT_NUMBER_9C8BFC]
14:08:51  INFO     [LLM SEES]  'account: [ACCOUNT_NUMBER_9C8BFC] — home loan outstanding?'
14:08:51  INFO     [RAG SEES]  'account: account balance — home loan outstanding?'
14:08:51  INFO     [TOKEN MAP] {'[ACCOUNT_NUMBER_9C8BFC]': '0101234567890'}  <- real number, memory only
14:08:51  INFO     [NODE] guard_node  DONE
14:08:51  INFO     [ROUTER] token_map=True  =>  cbs_node
14:08:51  INFO     [NODE] cbs_node  START
14:08:51  INFO     [CBS CALL]  cbs_lookup('0101****')
14:08:51  INFO     [CBS OK]    Dipak Shrestha | Loan | Rs. -3,200,000.00
14:08:51  INFO     [NODE] cbs_node  DONE


USER: account: 0101234567890 — home loan outstanding?

  AUDIT LOG — what each node saw:
  --------------------------------------------------------
  [guard_node]
    user_input : account: 0101234567890 — home loan outstanding?
    llm_query : account: [ACCOUNT_NUMBER_9C8BFC] — home loan outstanding?
    rag_query : account: account balance — home loan outstanding?
    token_map : [ACCOUNT_NUMBER_9C8BFC]  ->  "0101234567890"  <- REAL NUMBER
    // DETERMINISTIC — zero LLM. Account number in token_map only.
  [cbs_node]
    // Retrieved real number from token_map -> called CBS. NOT from LLM query.
    cbs_arg : 0101*********

  RESPONSE:
  ### Account Details
  **Holder:** Dipak Shrestha
  **Type:** Loan — Home Loan
  **Branch:** Thamel Branch
  **Status:** Active
  
  ### Balance
  **Current:** Rs. -3,200,000.00
  **Available:** Rs. 0.00
  
  ### Loan Info
  **Outstanding:** Rs. 3,200,000.00
  **Next EMI:** 2026-04-07
  **EMI:** Rs. 28,500.00



In [17]:
ACCOUNT = "1234567890"

r = graph.invoke({
    "user_input" : f"My account number is {ACCOUNT}. What is my balance?",
    "llm_query"  : "",
    "rag_query"  : "",
    "token_map"  : {},
    "intent"     : "",
    "response"   : "",
    "audit_log"  : [],
})

print("\nINVARIANT PROOF")
print("=" * 60)
print(f"Account number under test: {ACCOUNT}")
print()

checks = [
    ("user_input", r["user_input"], True,  "User typed it              (expected present)"),
    ("llm_query",  r["llm_query"],  False, "What LLM received          (must NOT be here)"),
    ("rag_query",  r["rag_query"],  False, "What vector store got      (must NOT be here)"),
    ("response",   r["response"],   False, "What user sees in response (must NOT be here)"),
]

all_ok = True
for field, value, should, label in checks:
    present = ACCOUNT in value
    if should:
        sym = "PRESENT  (expected)" if present else "MISSING — check code"
    else:
        sym = "NOT PRESENT  PASS" if not present else "LEAKED — INVARIANT VIOLATED!"
        if present:
            all_ok = False
    print(f"  {label:<44}  [{field:<12}]  {sym}")

print()
tok, real = list(r["token_map"].items())[0]
print(f"  token_map  :  {tok}")
print(f"               ->  \"{real}\"  (in-memory only, never logged)")
print()
print("=" * 60)
if all_ok:
    print("ALL INVARIANT CHECKS PASSED")
    print("Account number is in token_map only.")
    print("LLM, vector store, and response are all clean.")
else:
    print("INVARIANT VIOLATED — see output above")

14:08:51  INFO     ==================================================
14:08:51  INFO     [NODE] guard_node  START
14:08:51  INFO     [INPUT]     'My account number is 1234567890. What is my balance?'
14:08:51  INFO     [TOKENIZED]  '1234567890'  ->  [ACCOUNT_NUMBER_E807F1]
14:08:52  INFO     [LLM SEES]  'My account number is [ACCOUNT_NUMBER_E807F1]. What is my balance?'
14:08:52  INFO     [RAG SEES]  'My account number is account balance. What is my balance?'
14:08:52  INFO     [TOKEN MAP] {'[ACCOUNT_NUMBER_E807F1]': '1234567890'}  <- real number, memory only
14:08:52  INFO     [NODE] guard_node  DONE
14:08:52  INFO     [ROUTER] token_map=True  =>  cbs_node
14:08:52  INFO     [NODE] cbs_node  START
14:08:52  INFO     [CBS CALL]  cbs_lookup('1234****')
14:08:52  INFO     [CBS OK]    Ram Prasad Sharma | Savings | Rs. 125,430.50
14:08:52  INFO     [NODE] cbs_node  DONE



INVARIANT PROOF
Account number under test: 1234567890

  User typed it              (expected present)  [user_input  ]  PRESENT  (expected)
  What LLM received          (must NOT be here)  [llm_query   ]  NOT PRESENT  PASS
  What vector store got      (must NOT be here)  [rag_query   ]  NOT PRESENT  PASS
  What user sees in response (must NOT be here)  [response    ]  NOT PRESENT  PASS

  token_map  :  [ACCOUNT_NUMBER_E807F1]
               ->  "1234567890"  (in-memory only, never logged)

ALL INVARIANT CHECKS PASSED
Account number is in token_map only.
LLM, vector store, and response are all clean.


In [20]:
g = run_guardrail("Call me at 9841234567 or email me at ram@gmail.com")
print(g["llm_query"])

14:28:47  INFO     [INPUT]     'Call me at 9841234567 or email me at ram@gmail.com'
14:28:48  INFO     [LLM SEES]  'Call me at [PHONE] or email me at [EMAIL]'
14:28:48  INFO     [RAG SEES]  'Call me at [PHONE] or email me at [EMAIL]'
14:28:48  INFO     [TOKEN MAP] {}  (no account number detected)


Call me at [PHONE] or email me at [EMAIL]
